# Session 2 — Statsmodels: OLS Regression & Interpretation
**Phase 3+4 Combined | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

OLS regression is the workhorse of empirical finance:
- Factor models (CAPM, Fama-French 3-factor, Carhart 4-factor)
- Beta estimation
- Return attribution (how much of my return is market, how much is alpha?)
- Pairs trading (cointegration testing)
- Signal validation (does this factor predict returns?)

> 💡 The output of `statsmodels.OLS` is dense with numbers. Knowing which ones matter and which to ignore is the skill. Most people stare at R² — experienced analysts look at t-stats, residual diagnostics, and DW statistic first.


---
## 1️⃣ CAPM Regression — The Canonical Example

**Capital Asset Pricing Model (CAPM):**
```
R_asset - R_f = alpha + beta × (R_market - R_f) + epsilon
```

| Term | Meaning |
|------|---------|
| `alpha` | Excess return unexplained by market exposure (intercept) |
| `beta` | Sensitivity to market moves (slope) |
| `R_f` | Risk-free rate (usually T-bill rate) |
| `epsilon` | Residual (idiosyncratic risk) |

**Reading the OLS output:**

| Statistic | What it means | Good value? |
|-----------|--------------|-------------|
| R² | % of variance explained by the model | Higher is better (but not everything) |
| Adj. R² | R² penalised for extra variables | Use this over R² in multi-factor |
| t-statistic | Signal-to-noise for each coefficient | \|t\| > 2 ≈ significant at 5% |
| p-value | Probability coeff = 0 under H₀ | p < 0.05 to claim significance |
| F-statistic | Joint test: are ALL coefficients zero? | p < 0.05 = at least one matters |
| DW statistic | Durbin-Watson: autocorrelation in residuals | Close to 2 = no autocorrelation |


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# Download assets for regression
tickers = ['AAPL', 'SPY']
data = yf.download(tickers, start='2018-01-01', auto_adjust=True)['Close']
returns = np.log(data / data.shift(1)).dropna()

# CAPM regression: AAPL excess returns ~ SPY excess returns
# Approximate risk-free rate: 2% annual = 0.02/252 daily
rf_daily = 0.02 / 252

aapl_excess = returns['AAPL'] - rf_daily
spy_excess  = returns['SPY']  - rf_daily

# Add constant for intercept (alpha)
X = sm.add_constant(spy_excess)
y = aapl_excess

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# Extract and interpret the key numbers
alpha = model.params['const']
beta  = model.params['SPY']

alpha_annualised = alpha * 252
alpha_t  = model.tvalues['const']
beta_t   = model.tvalues['SPY']
alpha_p  = model.pvalues['const']
r2       = model.rsquared
dw       = sm.stats.stattools.durbin_watson(model.resid)

print("=== Key Metrics ===")
print(f"Alpha (daily):      {alpha:.6f}")
print(f"Alpha (annualised): {alpha_annualised*100:.2f}%")
print(f"Alpha t-stat:       {alpha_t:.3f}  (|t|>2 → significant)")
print(f"Alpha p-value:      {alpha_p:.4f}")
print()
print(f"Beta:               {beta:.3f}  (1.0 = same as market, >1 = amplified)")
print(f"Beta t-stat:        {beta_t:.3f}")
print()
print(f"R²:                 {r2:.4f}  ({r2*100:.1f}% of AAPL variance explained by market)")
print(f"Durbin-Watson:      {dw:.3f}  (2.0 = ideal, <2 = pos autocorrelation)")
print()
print("Interpretation:")
print(f"  Alpha {'IS' if alpha_p < 0.05 else 'is NOT'} statistically significant at 5%")
print(f"  Beta={beta:.2f} means AAPL moves ~{beta:.2f}x the market on average")
print(f"  The remaining {(1-r2)*100:.1f}% is idiosyncratic (company-specific) risk")


---
## 2️⃣ Fama-French 3-Factor Model

CAPM only uses one factor (market). Fama & French showed that **size** (SMB) and **value** (HML) explain additional return variation.

```
R - Rf = alpha + beta_mkt × MKT + beta_smb × SMB + beta_hml × HML + epsilon
```

| Factor | Meaning | Long | Short |
|--------|---------|------|-------|
| MKT | Market risk premium | All stocks | Risk-free |
| SMB | Small Minus Big | Small cap | Large cap |
| HML | High Minus Low (book/market) | Value stocks | Growth stocks |

> 💡 For a tech stock like AAPL, you'd expect: positive MKT beta, negative SMB (it's large cap), negative HML (it's a growth stock). Checking this is a good sanity test.


In [ ]:
# Fama-French factor data (manual construction — simplified)
# In production: download from Ken French's data library
# https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html

# For demo: use size and value proxies from ETFs
# IWM = small cap (Russell 2000), IWB = large cap (Russell 1000)
# IVE = value, IVW = growth

ff_tickers = ['AAPL', 'SPY', 'IWM', 'IWB', 'IVE', 'IVW']
ff_data = yf.download(ff_tickers, start='2018-01-01', auto_adjust=True)['Close']
ff_ret  = np.log(ff_data / ff_data.shift(1)).dropna()

# Construct factor proxies
mkt = ff_ret['SPY'] - rf_daily          # Market factor
smb = ff_ret['IWM'] - ff_ret['IWB']    # Small minus Big (size factor)
hml = ff_ret['IVE'] - ff_ret['IVW']    # Value minus Growth (value factor)
target = ff_ret['AAPL'] - rf_daily      # AAPL excess return

# 3-factor regression
X_ff = sm.add_constant(pd.DataFrame({'MKT': mkt, 'SMB': smb, 'HML': hml}))
X_ff = X_ff.dropna()
y_ff = target.reindex(X_ff.index)

model_ff = sm.OLS(y_ff, X_ff).fit()
print(model_ff.summary())


In [ ]:
# Compare: CAPM vs 3-Factor for AAPL
print("=== Model Comparison ===")
print(f"CAPM:      R²={model.rsquared:.4f}, Alpha t-stat={model.tvalues['const']:.3f}")
print(f"3-Factor:  R²={model_ff.rsquared:.4f}, Alpha t-stat={model_ff.tvalues['const']:.3f}")
print()
print("3-Factor Loadings:")
for factor in ['MKT', 'SMB', 'HML']:
    coef = model_ff.params[factor]
    tval = model_ff.tvalues[factor]
    pval = model_ff.pvalues[factor]
    sig  = '✅ significant' if pval < 0.05 else '(not significant)'
    print(f"  {factor}: {coef:.3f}  (t={tval:.2f}) {sig}")

print()
print("Sanity check for AAPL (large cap growth):")
print("  SMB should be negative (large cap → loads negatively on small-minus-big)")
print("  HML should be negative (growth → loads negatively on value-minus-growth)")


---
## 3️⃣ Regression Diagnostics — Don't Skip These

Running OLS without checking assumptions is like building on sand.

| Assumption | Diagnostic | What to look for |
|------------|-----------|-----------------|
| Linearity | Residuals vs fitted plot | Random scatter, no pattern |
| No autocorrelation | Durbin-Watson | Close to 2.0 |
| Homoskedasticity | Residuals vs time | No volatility clustering |
| Normality of residuals | Q-Q plot | Points on the diagonal |
| No multicollinearity | VIF (Variance Inflation Factor) | VIF < 10 per variable |


In [ ]:
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Residual analysis
resid = model.resid

print("=== Regression Diagnostics ===")
print()

# Durbin-Watson
dw = durbin_watson(resid)
print(f"Durbin-Watson:  {dw:.4f}")
print(f"  {'✅ Close to 2 — no obvious autocorrelation' if 1.5 < dw < 2.5 else '⚠️ Autocorrelation detected — residuals are not independent'}")
print()

# Ljung-Box test for autocorrelation in residuals
lb = acorr_ljungbox(resid, lags=[10], return_df=True)
lb_p = lb['lb_pvalue'].values[0]
print(f"Ljung-Box (10 lags): p={lb_p:.4f}")
print(f"  {'✅ No significant autocorrelation in residuals' if lb_p > 0.05 else '⚠️ Significant autocorrelation — model may be misspecified'}")
print()

# Residual normality
stat_sw, p_sw = __import__('scipy').stats.shapiro(resid.sample(500, random_state=42))
print(f"Shapiro-Wilk (residuals): p={p_sw:.6f}")
print(f"  {'⚠️ Residuals are not normal — standard errors may be unreliable' if p_sw < 0.05 else '✅ Residuals approximately normal'}")
print()

# Heteroskedasticity (Breusch-Pagan)
from statsmodels.stats.diagnostic import het_breuschpagan
_, bp_p, _, _ = het_breuschpagan(resid, model.model.exog)
print(f"Breusch-Pagan (heteroskedasticity): p={bp_p:.4f}")
print(f"  {'⚠️ Heteroskedastic residuals — vol clustering present' if bp_p < 0.05 else '✅ Homoskedastic residuals'}")


---
## ✅ Self-Check Questions

1. What does an alpha t-stat of 2.5 mean?
   > *The alpha estimate is 2.5 standard errors above zero. Strong evidence it's not due to chance. |t| > 2 is conventional threshold for 5% significance.*

2. Why is R² not sufficient to evaluate a regression model?
   > *High R² doesn't mean causation, correct specification, or useful predictions. A model can have R²=0.95 with terrible out-of-sample performance.*

3. What does a Durbin-Watson statistic of 1.1 indicate?
   > *Positive autocorrelation in residuals — consecutive residuals move together. Violates OLS assumption, inflates apparent significance.*

4. In Fama-French, what would a negative HML loading tell you about a stock?
   > *The stock behaves more like a growth stock (low book-to-market) than a value stock. Expected for tech firms.*

---
## 🧪 Optional Tasks

- Run a CAPM regression for 5 different stocks (e.g., TSLA, JNJ, GLD, TLT, AMZN). Which has the highest beta? Which has significant alpha?
- Compare R² in CAPM vs 3-factor for the same stock. How much extra variance do the additional factors explain?
- Plot residuals vs fitted values for your AAPL regression. Do you see volatility clustering? What does that imply?
- Run a regression for two periods: 2018-2020 vs 2021-2023. Does beta change? What does that mean for hedging?
